<a href="https://colab.research.google.com/github/Shishir120/ai-lab/blob/master/lab4-cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab-4 CNN**

**Name**: Shishir Pandeya

**Roll**: ACE079BCT062


A Convolutional Neural Network (CNN) is a type of deep learning model mainly used for image processing and computer vision tasks. It automatically detects important features (edges, shapes, textures) and works well with images and grid-like data. It also uses fewer parameters than fully connected networks for images making it best choice for image processing.

CNN mainly has 3 layers:

**1. Convolution Layer:**
-> This layer applies small filters(kernels) to the image to detect features like edges and patterns.

**2. Pooling Layer:**
-> This layer reduces the image size, keeps important information and makes the model faster.

**3. Fully Connected Layer:**
-> It is the final decision making layer.


# **1.Setup and Data Preparation**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Download and load datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

Using device: cuda


In [2]:
print(f"Dataloaders: {trainloader, testloader}")
print(f"Length of train dataloader: {len(trainloader)}")
print(f"Length of test dataloader: {len(testloader)}")

Dataloaders: (<torch.utils.data.dataloader.DataLoader object at 0x78992adfb860>, <torch.utils.data.dataloader.DataLoader object at 0x789a66122210>)
Length of train dataloader: 782
Length of test dataloader: 157


# **2. Model Building**

In [3]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        # 3 channels * 32 width * 32 height = 3072 input features
        self.flatten = nn.Flatten()
        self.fc_layers = nn.Sequential(
            nn.Linear(3072, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 10) # 10 output classes
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.fc_layers(x)

In [4]:
class AdaptedAlexNet(nn.Module):
    def __init__(self):
        super(AdaptedAlexNet, self).__init__()
        self.features = nn.Sequential(
            # Modified first layer: smaller kernel, smaller stride
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # Image size: 16x16

            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # Image size: 8x8

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), # Image size: 4x4
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            # Input size changed from ImageNet's 9216 to CIFAR's 4096 (256 channels * 4 * 4)
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [5]:
class TinyVGG(nn.Module):
    def __init__(self):
        super(TinyVGG, self).__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=10, out_channels=10, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Image size: 16x16
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=10, out_channels=10, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=10, out_channels=10, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Image size: 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(10 * 8 * 8, 10) # 10 channels * 8 * 8
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x

In [6]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"NN Parameters: {count_parameters(SimpleNN())}")
print(f"Adapted AlexNet Parameters: {count_parameters(AdaptedAlexNet())}")
print(f"TinyVGG Parameters: {count_parameters(TinyVGG())}")

NN Parameters: 1640330
Adapted AlexNet Parameters: 6976842
TinyVGG Parameters: 9420


# **3. Training Function**

In [7]:
import time
from sklearn.metrics import classification_report
import torch.nn as nn
import torch.optim as optim
import torch

def train_and_evaluate(model, trainloader, testloader, epochs=10, learning_rate=0.001):
    # 1. Setup Loss function and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # 2. Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    print(f"\n========== Training {model.__class__.__name__} ==========")
    start_time = time.time() # Start the stopwatch

    # 3. The Training Loop
    for epoch in range(epochs):
        model.train() # Set model to training mode (turns on Dropout)
        running_loss = 0.0

        for inputs, labels in trainloader:
            # Move data to GPU
            inputs, labels = inputs.to(device), labels.to(device)

            # The 5 Core Steps of PyTorch Training:
            optimizer.zero_grad()         # A. Clear old gradients
            outputs = model(inputs)       # B. Forward pass (guess the class)
            loss = criterion(outputs, labels) # C. Calculate the error (loss)
            loss.backward()               # D. Backward pass (calculate gradients)
            optimizer.step()              # E. Update the weights

            running_loss += loss.item()

        # Print average loss for this epoch
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(trainloader):.4f}")

    end_time = time.time() # Stop the stopwatch
    training_time = end_time - start_time
    print(f"-> Total Training Time: {training_time:.2f} seconds")

    # 4. The Evaluation Loop (Testing on unseen data)
    model.eval() # Set model to testing mode (turns off Dropout)
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): # Disable gradient tracking to save memory and speed up testing
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            # Get the index of the highest prediction score
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Save predictions for Scikit-Learn metrics
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    print(f"-> Test Accuracy: {accuracy:.2f}%\n")

    # 5. Extra Metrics: Precision, Recall, F1-Score
    print("Classification Report:")
    classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    print(classification_report(all_labels, all_preds, target_names=classes))
    print("====================================================\n")

    return training_time, accuracy

# **4. Execute the Training**

In [8]:
# 1. Instantiate the models
nn_model = SimpleNN()
alexnet_model = AdaptedAlexNet()
tinyvgg_model = TinyVGG()

# 2. Run the experiments (Using 10 epochs for a good baseline)
print("Starting experiments... Please wait, this will take a few minutes depending on your GPU.\n")

# Train Neural Network
time_nn, acc_nn = train_and_evaluate(nn_model, trainloader, testloader, epochs=10)

# Train Adapted AlexNet
time_alex, acc_alex = train_and_evaluate(alexnet_model, trainloader, testloader, epochs=10)

# Train TinyVGG
time_vgg, acc_vgg = train_and_evaluate(tinyvgg_model, trainloader, testloader, epochs=10)



Starting experiments... Please wait, this will take a few minutes depending on your GPU.


========== Training SimpleNN ==========
Epoch 1/10 - Loss: 1.6419
Epoch 2/10 - Loss: 1.4326
Epoch 3/10 - Loss: 1.3254
Epoch 4/10 - Loss: 1.2397
Epoch 5/10 - Loss: 1.1621
Epoch 6/10 - Loss: 1.0911
Epoch 7/10 - Loss: 1.0265
Epoch 8/10 - Loss: 0.9616
Epoch 9/10 - Loss: 0.8963
Epoch 10/10 - Loss: 0.8391
-> Total Training Time: 135.05 seconds
-> Test Accuracy: 53.55%

Classification Report:
              precision    recall  f1-score   support

       plane       0.61      0.62      0.61      1000
         car       0.69      0.54      0.60      1000
        bird       0.47      0.38      0.42      1000
         cat       0.35      0.40      0.37      1000
        deer       0.47      0.45      0.46      1000
         dog       0.47      0.39      0.43      1000
        frog       0.56      0.65      0.60      1000
       horse       0.60      0.60      0.60      1000
        ship       0.62      0.70

##Conclusion


In [9]:
print(f"Simple NN    | Time: {time_nn:.2f}s | Accuracy: {acc_nn:.2f}%")
print(f"AlexNet      | Time: {time_alex:.2f}s | Accuracy: {acc_alex:.2f}%")
print(f"TinyVGG      | Time: {time_vgg:.2f}s | Accuracy: {acc_vgg:.2f}%")

Simple NN    | Time: 135.05s | Accuracy: 53.55%
AlexNet      | Time: 155.29s | Accuracy: 76.99%
TinyVGG      | Time: 155.83s | Accuracy: 64.93%


# **5. Discussion**

## **Simple Neural Network (NN):**
Accuracy: 53.55%

Time: 135.05 seconds

This model was the fastest to train, but it had the worst accuracy. This makes sense. To use a standard NN, we have to flatten the 2D images into a 1D line. This destroys the spatial layout of the pixels. The network just couldn't learn the visual patterns effectively.

## **Adapted AlexNet:**
Accuracy: 76.99%

Time: 155.29 seconds

AlexNet was the clear winner for accuracy. Its deep convolutional layers learned complex visual features really well. However, it has a massive number of parameters. Because of this, it took the longest to train. The Dropout layers we added were crucial here to stop it from overfitting the training data.

## **TinyVGG:**
Accuracy: 64.93%

Time: 155.83 seconds

TinyVGG is a great middle ground. It used CNN layers to beat the Simple NN by a wide margin. Plus, it trained almost as fast as the Simple NN because it is much smaller than AlexNet. It proves you don't always need a huge model to get decent results.


# **Evaluation Metrics:**

Even though CIFAR-10 is perfectly balanced, some classes look very similar like cats and dogs, or trucks and cars.

Recall-> It tells us if the model is systematically missing a specific class.

Precision-> It tells us if the model is blindly guessing a certain class when it gets confused.

F1-Score-> It balances both of these. It gives us a much more honest look at how the model handles tricky images.


# **How AlexNet Improved:**

AlexNet was a huge step up from older architectures like LeNet. When reading the paper, a few major breakthroughs stand out:

**ReLU Activation:** Older models used Tanh or Sigmoid functions, which slowed down learning. AlexNet used ReLU. This solved the "vanishing gradient" problem and made training much faster.

**Dropout:** Massive models tend to just memorize the training data. AlexNet introduced Dropout, which randomly turns off neurons during training. This forces the network to learn stronger, more general features.

**Overlapping Max Pooling:** Instead of standard pooling, AlexNet used overlapping windows to shrink the image. The authors found this slightly reduced error rates and helped fight overfitting.

**Built for GPUs:** AlexNet was specifically designed to be split across parallel GPUs. This is what finally allowed researchers to train deep networks on millions of images.


# **6.Conclusion:**

This lab clearly shows why Convolutional Neural Networks (CNNs) are the standard for computer vision.

The Simple NN failed to understand the 2D structure of the images, scoring a low 53.55%. On the other hand, the CNN architectures (TinyVGG and AlexNet) performed much better. TinyVGG showed that a lightweight CNN can still get a solid 64.93%.

AlexNet's deeper architecture like ReLU and Dropout pushed the accuracy up to an impressive 76.74%. Ultimately, TinyVGG is best if the need fast efficiency, while AlexNet is the better choice for maximum predictive power.